# CoAtNet Baseline and 672 × 672 Fine-Tuning

This notebook contains three stages:

1. Download and extract the ISIC 2019 training-image archive.
2. Train and evaluate a CoAtNet model with auxiliary SCC, AK, and MEL prediction heads; generate calibration metrics, feature exports, and Grad-CAM++ visualizations.
3. Fine-tune the Stage 1 checkpoint at 672 × 672 resolution and repeat evaluation.

## Before running

- Obtain the ISIC 2019 images and the train/validation/test split CSV files from their authorized sources. The dataset and split files are **not** included in this repository.
- Set `PROJECT_ROOT`, `DATASET_CSV_DIR`, and `IMAGE_DIR` to paths available in your environment, or edit the defaults in the configuration below.
- Install the notebook dependencies in a controlled environment. A repository-level `requirements.txt` should be maintained separately.
- Run the cells in order. Stage 2 depends on variables and classes defined in Stage 1 and on the Stage 1 checkpoint.
- Outputs are intentionally cleared in the repository copy. The code can regenerate reports and figures when executed.

## Reproducibility note

The resolution protocol in this notebook is intentional: Stage 1 uses `CFG.IMG_SIZE = 224` for baseline training, and Stage 2 fine-tunes the Stage 1 checkpoint using `FT.IMG_SIZE = 672`. Ensure that this two-stage setup matches the experiment reported in the manuscript.


## Stage 0 — Download and extract ISIC 2019 images

This cell downloads the ISIC 2019 training-image ZIP archive from the configured public archive URL and extracts it under `/content`. It does not download the train/validation/test split CSV files.

The image dataset is not committed to GitHub. Check the dataset terms and the official source before use. If the target directory already exists, the extraction step is skipped.


In [ ]:
# ============================================================
# ISIC 2019: Download the training images archive and extract it to /content/ISIC_2019_Training_Input.
# ============================================================
import os, zipfile, requests
from tqdm.auto import tqdm

URL = "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip"
ZIP_PATH = "/content/ISIC_2019_Training_Input.zip"
EXTRACT_DIR = "/content"  # The archive is expected to contain an ISIC_2019_Training_Input directory.

os.makedirs(EXTRACT_DIR, exist_ok=True)

# -------- Download with a progress bar --------
if not os.path.exists(ZIP_PATH):
    print("Starting download:", URL)
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        pbar = tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc="Downloading")
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
        pbar.close()
else:
    print("Archive already exists:", ZIP_PATH)

# -------- Extract with a progress bar --------
target_path = os.path.join(EXTRACT_DIR, "ISIC_2019_Training_Input")
if not os.path.exists(target_path):
    print("Extracting archive to:", EXTRACT_DIR)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        members = zf.infolist()
        pbar = tqdm(total=len(members), desc="Extracting", unit="file")
        for m in members:
            zf.extract(m, EXTRACT_DIR)
            pbar.update(1)
        pbar.close()
else:
    print("Target directory already exists:", target_path)

# -------- Final status --------
print("\nDataset is ready:", target_path)
print("First few entries:", sorted(os.listdir(target_path))[:5])


## Stage 1 — CoAtNet baseline training and evaluation

This cell defines the dataset, augmentation pipeline, class-balancing strategy, CoAtNet backbone, auxiliary classification heads, training loop, validation-based checkpoint selection, test evaluation, temperature scaling, feature export, and Grad-CAM++ visualization.

The code saves checkpoints and generated artifacts under `CFG.ROOT_DIR`. Keep generated checkpoints, full feature arrays, predictions, and large visualizations out of the Git repository unless there is a specific, justified need to publish them.

**Resolution protocol:** Stage 1 baseline training uses `CFG.IMG_SIZE = 224`; Stage 2 fine-tuning uses `FT.IMG_SIZE = 672`. Keep this distinction consistent with the experiment reported in the manuscript.


In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# CoAtNet baseline: training, evaluation, calibration, feature export, and Grad-CAM++
# RESOLUTION PROTOCOL: Stage 1 baseline training uses CFG.IMG_SIZE = 224. Stage 2 fine-tuning uses FT.IMG_SIZE = 672. Ensure that this two-stage setup matches the experiment reported in the manuscript.
# - Project outputs are written under CFG.ROOT_DIR.
# - Training: class-balanced focal loss, deferred re-weighting (DRW), and auxiliary SCC/AK/MEL heads (lambda=0.35)
# - Stabilization: EMA, warmup + cosine schedule, channels_last, gradient clipping, and deterministic algorithms
# - Class balancing: WeightedRandomSampler combined with class-balanced focal loss
# - Evaluation: validation and test reports, raw/normalized confusion matrices, ROC-AUC, MCC, and macro-F1
# - Calibration: select the temperature on validation data, apply it to test probabilities, and report ECE/Brier scores
# - Test-time augmentation: original image plus horizontal and vertical flips
# - Explainability: Grad-CAM++ examples from validation and test splits
# - Outputs: checkpoints, figures, reports, features, and CAM images
# ============================================================

# Install compatible dependencies from the repository-level requirements.txt before running this notebook.

import os, math, random, zipfile, warnings, json, datetime, platform
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from tqdm.auto import tqdm

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, matthews_corrcoef, log_loss
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

# ---------------- CUDA and determinism ----------------
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False  # Disable cuDNN benchmarking for more deterministic execution
torch.use_deterministic_algorithms(True, warn_only=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- Configuration ----------------
class CFG:
    # Model and data
    MODEL_NAME = "coatnet_1_rw_224"      # Alternative: "coatnet_2_rw_224"
    IMG_SIZE   = 224
    NUM_CLASSES = 8
    CLASS_NAMES = ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']

    # Project root and output directories
    ROOT_DIR = os.environ.get("PROJECT_ROOT", "/content/skin_lesion_calibrated_ensemble/coatnet_baseline")
    CKPT_DIR = os.path.join(ROOT_DIR, "ckpt")
    FIG_DIR  = os.path.join(ROOT_DIR, "figs")
    REP_DIR  = os.path.join(ROOT_DIR, "reports")
    FEAT_DIR = os.path.join(ROOT_DIR, "features")
    CAM_DIR  = os.path.join(ROOT_DIR, "cams")

    # Dataset paths
    DATASET_CSV_DIR = os.environ.get("DATASET_CSV_DIR", "/content/skin_lesion_calibrated_ensemble/splits")
    IMAGE_DIR = os.environ.get("IMAGE_DIR", "/content/ISIC_2019_Training_Input")
    TRAIN_CSV = "train_split.csv"; VAL_CSV = "val_split.csv"; TEST_CSV = "test_split.csv"

    # Training
    EPOCHS = 200
    BATCH_SIZE = 16
    LR = 3e-5
    WD = 1e-4
    PATIENCE = 25
    WARMUP_EPOCHS = 5
    DRW_SWITCH = 8
    EMA_DECAY = 0.9996
    GRAD_CLIP = 1.0
    DROP_PATH = 0.2
    GAMMA = 2.5
    AUX_LAMBDA = 0.35
    NUM_WORKERS = 8

    # Evaluation
    TTA = True
    CAM_PER_CLASS = 20  # Number of CAM examples to save per class

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "cuda": torch.cuda.manual_seed_all(SEED)

# ---------------- Drive (Colab) ----------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

# ---------------- Create output directories ----------------
for d in [CFG.ROOT_DIR, CFG.CKPT_DIR, CFG.FIG_DIR, CFG.REP_DIR, CFG.FEAT_DIR, CFG.CAM_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"DEVICE: {DEVICE} | MODEL: {CFG.MODEL_NAME} | IMG: {CFG.IMG_SIZE}")
print(f"ROOT DIR: {CFG.ROOT_DIR}")

# ---------------- Dataset ----------------
class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, return_id=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tfms = transforms
        self.return_id = return_id

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'] + ".jpg")
        image = np.array(Image.open(img_path).convert('RGB'))
        label = int(row['label'])
        if self.tfms: image = self.tfms(image=image)['image']
        if self.return_id:
            return image, label, row['image']
        return image, label

# ---- Augmentations ----
train_tfms = A.Compose([
    A.RandomResizedCrop(size=(CFG.IMG_SIZE, CFG.IMG_SIZE), scale=(0.90, 1.0), ratio=(0.9, 1.1)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.HueSaturationValue(10, 15, 10, p=0.3),
    A.CoarseDropout(max_holes=2, max_height=int(CFG.IMG_SIZE*0.05), max_width=int(CFG.IMG_SIZE*0.05), p=0.25),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])
val_tfms = A.Compose([
    A.Resize(height=CFG.IMG_SIZE, width=CFG.IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

# ---------------- CSV & Sampler ----------------
train_df = pd.read_csv(os.path.join(CFG.DATASET_CSV_DIR, CFG.TRAIN_CSV))
val_df   = pd.read_csv(os.path.join(CFG.DATASET_CSV_DIR, CFG.VAL_CSV))
test_df  = pd.read_csv(os.path.join(CFG.DATASET_CSV_DIR, CFG.TEST_CSV))

cls2idx = {name:i for i,name in enumerate(CFG.CLASS_NAMES)}
for df in [train_df,val_df,test_df]:
    df['label'] = df['class_name'].map(cls2idx).astype(int)

class_counts = train_df['label'].value_counts().sort_index().values
class_weights_np = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights_np[train_df['label'].values]
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True
)

train_ds = ISICDataset(train_df, CFG.IMAGE_DIR, transforms=train_tfms)
val_ds   = ISICDataset(val_df,   CFG.IMAGE_DIR, transforms=val_tfms)
test_ds  = ISICDataset(test_df,  CFG.IMAGE_DIR, transforms=val_tfms)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)

# ---------------- Loss functions ----------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.5, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = (1-pt)**self.gamma * ce
        if self.alpha is not None:
            a = self.alpha.to(inputs.device)
            at = a.gather(0, targets)
            loss = at * loss
        return loss.mean() if self.reduction=='mean' else loss

bce_logits = nn.BCEWithLogitsLoss(reduction='mean')

def effective_num_alpha(labels, num_classes=8, beta=0.9999, boost=None):
    if boost is None: boost = {0:1.3, 4:1.25, 6:1.4}  # Emphasize AK, MEL, and SCC classes
    cnt = np.bincount(labels, minlength=num_classes).astype(np.int64)
    eff = (1.0 - np.power(beta, cnt)) / (1.0 - beta)
    w = 1.0 / np.maximum(eff, 1e-8)
    w = w / w.sum() * num_classes
    for k,v in boost.items(): w[k] *= v
    return torch.tensor(w, dtype=torch.float32)

alpha_eff = effective_num_alpha(train_df['label'].values, num_classes=CFG.NUM_CLASSES)
criterion_warm = FocalLoss(alpha=alpha_eff * 0.3, gamma=CFG.GAMMA)
criterion_cb   = FocalLoss(alpha=alpha_eff,         gamma=CFG.GAMMA)

def aux_targets_from_labels(labels):
    # labels: [B] 0..7; aux heads: SCC(6), AK(0), MEL(4)
    scc = (labels==6).float()
    ak  = (labels==0).float()
    mel = (labels==4).float()
    return scc, ak, mel

def total_loss(outputs, labels, crit_cls, aux_lambda=CFG.AUX_LAMBDA):
    logits, aux, _ = outputs
    loss_main = crit_cls(logits, labels)
    scc_t, ak_t, mel_t = aux_targets_from_labels(labels)
    loss_aux = (
        bce_logits(aux["scc"], scc_t) +
        bce_logits(aux["ak"],  ak_t)  +
        bce_logits(aux["mel"], mel_t)
    ) / 3.0
    return loss_main + aux_lambda * loss_aux, loss_main, loss_aux

# ---------------- Model and auxiliary heads ----------------
class CoAtNetWithAux(nn.Module):
    def __init__(self, base_name, num_classes=8, drop_path_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            base_name, pretrained=True, num_classes=0, global_pool='avg',
            drop_path_rate=drop_path_rate, img_size=CFG.IMG_SIZE
        )
        self.feat_dim = self.backbone.num_features
        self.head_main = nn.Linear(self.feat_dim, num_classes)
        self.head_scc  = nn.Linear(self.feat_dim, 1)
        self.head_ak   = nn.Linear(self.feat_dim, 1)
        self.head_mel  = nn.Linear(self.feat_dim, 1)
    def forward(self, x):
        feat = self.backbone(x)         # [B, D]
        logits = self.head_main(feat)
        aux = {
            "scc": self.head_scc(feat).squeeze(1),
            "ak":  self.head_ak(feat).squeeze(1),
            "mel": self.head_mel(feat).squeeze(1),
        }
        return logits, aux, feat

model = CoAtNetWithAux(CFG.MODEL_NAME, CFG.NUM_CLASSES, drop_path_rate=CFG.DROP_PATH).to(DEVICE)
model = model.to(memory_format=torch.channels_last)

# EMA and optimizer
from timm.utils import ModelEmaV2
ema = ModelEmaV2(model, decay=CFG.EMA_DECAY, device=DEVICE)

optimizer = AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
scaler = GradScaler()

total_steps = len(train_loader) * CFG.EPOCHS
warmup_steps = len(train_loader) * CFG.WARMUP_EPOCHS
def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    prog = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * prog))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

T1 = CFG.DRW_SWITCH
print(f"DRW switch @ epoch {T1}")

# ---------------- Training loop ----------------
history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_f1, patience = -1.0, 0

for epoch in range(CFG.EPOCHS):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.EPOCHS}")

    for imgs, labels in pbar:
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            outputs = model(imgs)
            crit_now = criterion_cb if epoch >= T1 else criterion_warm
            loss, loss_main, loss_aux = total_loss(outputs, labels, crit_now, CFG.AUX_LAMBDA)

        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        ema.update(model)

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # Validation (EMA)
    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE, memory_format=torch.channels_last)
            labels = labels.to(DEVICE)
            with autocast():
                logits, _, _ = ema.module(imgs)
                vloss = F.cross_entropy(logits, labels)
            val_loss += vloss.item()
            pred = logits.argmax(1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_f1  = f1_score(np.array(all_labels), np.array(all_preds), average="macro")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"[Epoch {epoch+1:03d}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_macroF1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1, patience = val_f1, 0
        torch.save(ema.module.state_dict(), os.path.join(CFG.CKPT_DIR, "best_model_stage1.pth"))
        print(f"  New best validation macro-F1: {best_f1:.4f}")
    else:
        patience += 1
        if patience >= CFG.PATIENCE:
            print("Early stopping triggered."); break

# ---------------- Plotting and metric helpers ----------------
def plot_and_save_cm(cm, labels, title, out_path):
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
                xticklabels=labels, yticklabels=labels)
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(out_path, dpi=300); plt.close()

def plot_and_save_cm_norm(cm, labels, title, out_path):
    cmn = cm.astype(float)/np.maximum(cm.sum(axis=1, keepdims=True), 1)
    plt.figure(figsize=(8,6))
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='magma',
                xticklabels=labels, yticklabels=labels, vmin=0, vmax=1)
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(out_path, dpi=300); plt.close()

def plot_and_save_roc(y_true, y_prob, labels, out_path, tag=""):
    y_bin = label_binarize(y_true, classes=np.arange(len(labels)))
    fpr, tpr, roc_auc = {}, {}, {}
    valid = []
    for i in range(len(labels)):
        pos = y_bin[:, i].sum(); neg = len(y_bin[:, i]) - pos
        if pos>0 and neg>0:
            fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i]); valid.append(i)
    plt.figure(figsize=(8,6))
    for i in valid:
        plt.plot(fpr[i], tpr[i], lw=1.4, label=f"{labels[i]} (AUC={roc_auc[i]:.3f})")
    plt.plot([0,1],[0,1],'k--',lw=1)
    try:
        micro_fpr, micro_tpr, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
        micro_auc = auc(micro_fpr, micro_tpr)
        plt.plot(micro_fpr, micro_tpr, lw=2.0, label=f"micro (AUC={micro_auc:.3f})")
    except ValueError:
        pass
    plt.title(f"ROC — {tag}"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(fontsize=8, ncol=2)
    plt.tight_layout(); plt.savefig(out_path, dpi=300); plt.close()

def metrics_table(y_true, y_prob):
    y_pred = y_prob.argmax(1)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        y_bin = label_binarize(y_true, classes=np.arange(CFG.NUM_CLASSES))
        auc_macro = roc_auc_score(y_bin, y_prob, average="macro")
        auc_weighted = roc_auc_score(y_bin, y_prob, average="weighted")
        auc_micro = roc_auc_score(y_bin.ravel(), y_prob.ravel())
    except Exception:
        auc_macro = auc_weighted = auc_micro = float("nan")
    return macro_f1, mcc, auc_micro, auc_macro, auc_weighted

@torch.no_grad()
def predict_loader(model, loader, tta=True, return_ids=False):
    model.eval()
    probs_all, labels_all, ids_all = [], [], []
    for batch in tqdm(loader, total=len(loader), desc="Predict"):
        if return_ids:
            imgs, labels, ids = batch
        else:
            imgs, labels = batch; ids = None
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last)
        labels = labels.to(DEVICE)
        with autocast():
            if tta:
                batch_cat = torch.cat([imgs, torch.flip(imgs, [3]), torch.flip(imgs, [2])], dim=0)
                logits = model(batch_cat)[0]
                p0, p1, p2 = logits.chunk(3, dim=0)
                probs = (p0.softmax(1) + p1.softmax(1) + p2.softmax(1)) / 3.0
            else:
                probs = model(imgs)[0].softmax(1)
        probs_all.append(probs.float().cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
        if return_ids and ids is not None: ids_all += list(ids)
    probs_all = np.concatenate(probs_all, axis=0)
    labels_all = np.array(labels_all, dtype=int)
    return probs_all, labels_all, ids_all

def expected_calibration_error(probs, labels, n_bins=15):
    bins = np.linspace(0,1,n_bins+1)
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    acc = (pred==labels).astype(float)
    ece = 0.0
    for i in range(n_bins):
        m = (conf>bins[i]) & (conf<=bins[i+1]); p = m.mean()
        if p>0: ece += abs(acc[m].mean() - conf[m].mean())*p
    return float(ece)

# ---------------- Load the best EMA checkpoint ----------------
ema.module.load_state_dict(torch.load(os.path.join(CFG.CKPT_DIR, "best_model_stage1.pth"), map_location=DEVICE))
best = ema.module.eval()

# ---------------- Validation/test evaluation and reports ----------------
# Create deterministic evaluation data loaders (image IDs are required)
val_ds_eval  = ISICDataset(val_df,  CFG.IMAGE_DIR, transforms=val_tfms,  return_id=True)
test_ds_eval = ISICDataset(test_df, CFG.IMAGE_DIR, transforms=val_tfms,  return_id=True)
val_loader_eval  = DataLoader(val_ds_eval,  batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
test_loader_eval = DataLoader(test_ds_eval, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)

# Validation split
probs_val, labels_val, ids_val = predict_loader(best, val_loader_eval, tta=CFG.TTA, return_ids=True)
preds_val = probs_val.argmax(1)
cr_val = classification_report(labels_val, preds_val, target_names=CFG.CLASS_NAMES, digits=4)
cm_val = confusion_matrix(labels_val, preds_val)

with open(os.path.join(CFG.REP_DIR, "classification_report_val.txt"), "w") as f: f.write(cr_val)
plot_and_save_cm(cm_val, CFG.CLASS_NAMES, "Confusion Matrix (VAL)", os.path.join(CFG.FIG_DIR, "cm_val_raw.png"))
plot_and_save_cm_norm(cm_val, CFG.CLASS_NAMES, "Confusion Matrix Normalized (VAL)", os.path.join(CFG.FIG_DIR, "cm_val_norm.png"))
plot_and_save_roc(labels_val, probs_val, CFG.CLASS_NAMES, os.path.join(CFG.FIG_DIR, "roc_val.png"), tag="VAL")

f1_v, mcc_v, auc_micro_v, auc_macro_v, auc_weighted_v = metrics_table(labels_val, probs_val)
pd.DataFrame([{"macro_f1": f1_v, "mcc": mcc_v, "auc_micro": auc_micro_v, "auc_macro": auc_macro_v, "auc_weighted": auc_weighted_v}])\
  .to_csv(os.path.join(CFG.REP_DIR, "val_summary.csv"), index=False)

pd.DataFrame({"image": ids_val, "label": labels_val,
              **{f"p_{c}": probs_val[:,i] for i,c in enumerate(CFG.CLASS_NAMES)}})\
  .to_csv(os.path.join(CFG.REP_DIR, "predictions_val.csv"), index=False)

# Temperature scaling (select T on validation data)
Ts = np.arange(0.5, 5.01, 0.1)
best_T, best_nll = 1.0, 1e9
eps = 1e-8
logp_val = np.log(np.maximum(probs_val, eps))
for T in Ts:
    scaled_logits = (logp_val / T).clip(-50, 50)
    scaled_logits = scaled_logits - scaled_logits.max(axis=1, keepdims=True)
    scaled_probs = np.exp(scaled_logits)
    scaled_probs = scaled_probs / scaled_probs.sum(axis=1, keepdims=True)
    nll = log_loss(labels_val, scaled_probs, labels=np.arange(CFG.NUM_CLASSES))
    if nll < best_nll: best_nll, best_T = nll, T

# Evaluate the test split; raw and calibrated metrics are saved separately, while plots show raw probabilities
probs_test, labels_test, ids_test = predict_loader(best, test_loader_eval, tta=CFG.TTA, return_ids=True)
preds_test = probs_test.argmax(1)
cr_test = classification_report(labels_test, preds_test, target_names=CFG.CLASS_NAMES, digits=4)
cm_test = confusion_matrix(labels_test, preds_test)

with open(os.path.join(CFG.REP_DIR, "classification_report_test.txt"), "w") as f: f.write(cr_test)
plot_and_save_cm(cm_test, CFG.CLASS_NAMES, "Confusion Matrix (TEST)", os.path.join(CFG.FIG_DIR, "cm_test_raw.png"))
plot_and_save_cm_norm(cm_test, CFG.CLASS_NAMES, "Confusion Matrix Normalized (TEST)", os.path.join(CFG.FIG_DIR, "cm_test_norm.png"))
plot_and_save_roc(labels_test, probs_test, CFG.CLASS_NAMES, os.path.join(CFG.FIG_DIR, "roc_test.png"), tag="TEST")

f1_t, mcc_t, auc_micro_t, auc_macro_t, auc_weighted_t = metrics_table(labels_test, probs_test)
pd.DataFrame([{"macro_f1": f1_t, "mcc": mcc_t, "auc_micro": auc_micro_t, "auc_macro": auc_macro_t, "auc_weighted": auc_weighted_t, "TTA": CFG.TTA}])\
  .to_csv(os.path.join(CFG.REP_DIR, "test_summary.csv"), index=False)

pd.DataFrame({"image": ids_test, "label": labels_test,
              **{f"p_{c}": probs_test[:,i] for i,c in enumerate(CFG.CLASS_NAMES)}})\
  .to_csv(os.path.join(CFG.REP_DIR, "predictions_test.csv"), index=False)

# Calibration metrics on the test split
probs_test_raw = probs_test.copy()
probs_test_cal = torch.tensor(np.log(np.maximum(probs_test_raw,eps))/best_T).softmax(dim=1).numpy()
labels_test_oh = np.eye(CFG.NUM_CLASSES)[labels_test]

ece_raw = expected_calibration_error(probs_test_raw, labels_test, 15)
ece_cal = expected_calibration_error(probs_test_cal, labels_test, 15)
brier_raw = float(np.mean((probs_test_raw - labels_test_oh)**2))
brier_cal = float(np.mean((probs_test_cal - labels_test_oh)**2))
with open(os.path.join(CFG.REP_DIR, "calibration.json"), "w") as f:
    json.dump({"best_T": float(best_T), "val_NLL": float(best_nll),
               "ECE_raw": ece_raw, "ECE_cal": ece_cal,
               "Brier_raw": brier_raw, "Brier_cal": brier_cal}, f, indent=2)

print(f"\nVAL Summary → Macro-F1: {f1_v:.4f} | MCC: {mcc_v:.4f} | AUC micro/macro/weighted: {auc_micro_v:.4f}/{auc_macro_v:.4f}/{auc_weighted_v:.4f}")
print(f"TEST Summary → Macro-F1: {f1_t:.4f} | MCC: {mcc_t:.4f} | AUC micro/macro/weighted: {auc_micro_t:.4f}/{auc_macro_t:.4f}/{auc_weighted_t:.4f}")
print(f"Calibration(T={best_T:.2f}) → ECE raw/cal: {ece_raw:.4f}/{ece_cal:.4f} | Brier raw/cal: {brier_raw:.4f}/{brier_cal:.4f}")

# ---------------- Feature export (train/validation/test) ----------------
@torch.no_grad()
def export_features(m, loader, split_name, with_ids=False):
    m.eval()
    feats, labels, ids = [], [], []
    for batch in tqdm(loader, desc=f"Export {split_name}"):
        if with_ids:
            imgs, lbls, idlist = batch
        else:
            imgs, lbls = batch; idlist=None
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last)
        with autocast():
            _, _, feat = m(imgs)
        feats.append(feat.float().cpu().numpy())
        labels.extend(lbls.numpy())
        if with_ids and idlist is not None: ids += list(idlist)
    feats = np.concatenate(feats, axis=0)
    np.save(os.path.join(CFG.FEAT_DIR, f"features_{split_name}.npy"), feats)
    meta = {"label": labels}
    if with_ids: meta["image"] = ids
    pd.DataFrame(meta).to_csv(os.path.join(CFG.FEAT_DIR, f"meta_{split_name}.csv"), index=False)
    print(f"✅ {split_name} features saved; shape={feats.shape}")

# Create loaders that also return image IDs for feature export
train_loader_eval = DataLoader(ISICDataset(train_df, CFG.IMAGE_DIR, transforms=val_tfms, return_id=True),
                               batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                               num_workers=CFG.NUM_WORKERS, pin_memory=True)
val_loader_eval2  = DataLoader(val_ds_eval,  batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                               num_workers=CFG.NUM_WORKERS, pin_memory=True)
test_loader_eval2 = DataLoader(test_ds_eval, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                               num_workers=CFG.NUM_WORKERS, pin_memory=True)

export_features(best, train_loader_eval, "train", with_ids=True)
export_features(best, val_loader_eval2,  "val",   with_ids=True)
export_features(best, test_loader_eval2, "test",  with_ids=True)

# ---------------- Grad-CAM++ examples ----------------
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# Automatically locate the last convolutional layer (a practical default for timm CoAtNet models)
def find_last_conv_layer(model):
    last_conv = None
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            last_conv = m
    if last_conv is None:
        raise RuntimeError("No convolutional layer was found. Select a target layer manually for CAM.")
    return last_conv

target_layer = find_last_conv_layer(best)

def tensor_to_img_np(t):
    # Input tensor shape: [3, H, W]. The tensor is normalized and must be converted back for visualization.
    # CAM overlays require an RGB image in the [0, 1] range; reconstruct an approximate display image.
    tn = t.clone().float().cpu()
    # Reverse ImageNet normalization
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    tn = (tn*std + mean).clamp(0,1)
    return np.transpose(tn.numpy(), (1,2,0))

class LogitsOnlyWrapper(nn.Module):
    """Expose only class logits to Grad-CAM implementations."""
    def __init__(self, wrapped_model):
        super().__init__()
        self.wrapped_model = wrapped_model

    def forward(self, x):
        return self.wrapped_model(x)[0]


def save_cams(split_name, loader, probs, labels, ids, per_class=20):
    os.makedirs(os.path.join(CFG.CAM_DIR, split_name), exist_ok=True)
    # Select the first N examples for each ground-truth class
    preds = probs.argmax(1)
    chosen = {k: [] for k in range(CFG.NUM_CLASSES)}
    # Collect selected indices
    for i, (y, p) in enumerate(zip(labels, preds)):
        c = int(y)
        if len(chosen[c]) < per_class:
            chosen[c].append(i)
        if all(len(v) >= per_class for v in chosen.values()):
            break

    # Iterate over the loader again and save CAMs for the selected indices
    cam_model = LogitsOnlyWrapper(best)
    cam = GradCAMPlusPlus(model=cam_model, target_layers=[target_layer], use_cuda=(DEVICE == "cuda"))
    idx_global = 0
    save_map = set(sum(chosen.values(), []))
    loader_plain = DataLoader(loader.dataset, batch_size=1, shuffle=False)
    for (img, y, idstr) in tqdm(loader_plain, total=len(loader_plain), desc=f"CAM {split_name}"):
        if idx_global in save_map:
            img = img.to(DEVICE)
            with torch.no_grad(), autocast():
                logits, _, _ = best(img)
                pred = logits.argmax(1).item()
            grayscale_cam = cam(input_tensor=img, targets=None)[0]  # [H, W]
            rgb_np = tensor_to_img_np(img[0])
            vis = show_cam_on_image(rgb_np, grayscale_cam, use_rgb=True)
            cls_name = CFG.CLASS_NAMES[int(y)]
            out_dir = os.path.join(CFG.CAM_DIR, split_name, cls_name)
            os.makedirs(out_dir, exist_ok=True)
            Image.fromarray(vis).save(os.path.join(out_dir, f"{idstr[0]}_pred-{CFG.CLASS_NAMES[pred]}.png"))
        idx_global += 1

# Generate CAM examples for validation and test splits
save_cams("val",  val_loader_eval,  probs_val,  labels_val,  ids_val,  per_class=CFG.CAM_PER_CLASS)
save_cams("test", test_loader_eval, probs_test, labels_test, ids_test, per_class=CFG.CAM_PER_CLASS)

# ---------------- Save software versions and run configuration ----------------
versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "timm": timm.__version__,
    "albumentations": A.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
}
with open(os.path.join(CFG.REP_DIR, "versions.json"), "w") as f:
    json.dump(versions, f, indent=2)

run_config = {
    "timestamp": datetime.datetime.now().isoformat(),
    "seed": SEED,
    "model": CFG.MODEL_NAME,
    "img_size": CFG.IMG_SIZE,
    "classes": CFG.CLASS_NAMES,
    "epochs": CFG.EPOCHS,
    "batch_size": CFG.BATCH_SIZE,
    "lr": CFG.LR,
    "weight_decay": CFG.WD,
    "patience": CFG.PATIENCE,
    "warmup_epochs": CFG.WARMUP_EPOCHS,
    "drw_switch": CFG.DRW_SWITCH,
    "ema_decay": CFG.EMA_DECAY,
    "grad_clip": CFG.GRAD_CLIP,
    "drop_path": CFG.DROP_PATH,
    "gamma": CFG.GAMMA,
    "aux_lambda": CFG.AUX_LAMBDA,
    "tta": CFG.TTA
}
with open(os.path.join(CFG.REP_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print("\nCompleted: training, evaluation reports, calibration metrics, Grad-CAM++ images, and feature exports are available.")
print(f"Reports: {CFG.REP_DIR}")
print(f"Figures: {CFG.FIG_DIR}")
print(f"Features: {CFG.FEAT_DIR}")
print(f"Checkpoint: {CFG.CKPT_DIR}/best_model_stage1.pth")
print(f"CAM images: {CFG.CAM_DIR}/(val|test)/<class>/image_pred-<class>.png")


## Stage 2 — Fine-tune at 672 × 672

Run this cell only after Stage 1 has completed successfully in the same notebook session. It reuses the Stage 1 data frames, dataset class, model class, loss functions, configuration, and checkpoint.

This stage rebuilds the CoAtNet backbone for 672 × 672 inputs, attempts to resize compatible relative-position bias tables, fine-tunes the model, selects the checkpoint using validation macro-F1, and produces test metrics, calibration metrics, feature exports, and CAM examples.

The calibration temperature is selected using validation data and then applied to the test probabilities. Do not use test-set metrics to select the checkpoint or calibration temperature.


In [ ]:
# ===================== STAGE 2: Fine-tuning at 672 x 672 =====================
# Run this cell after Stage 1 has completed successfully.
# This stage reuses classes and objects defined in Stage 1 (FocalLoss, total_loss, ISICDataset, CoAtNetWithAux, and the data splits).

import os, math, json, datetime, platform, warnings, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from timm.utils import ModelEmaV2
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, matthews_corrcoef, log_loss
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- Fine-tuning configuration (Stage 1 settings, with a larger image size and separate output paths) ----------
class FT:
    IMG_SIZE = 672
    EPOCHS = CFG.EPOCHS                # 200
    BATCH_SIZE = 12                    # Adjust to available GPU memory; Stage 1 used a batch size of 16
    LR = CFG.LR                        # 3e-5
    WD = CFG.WD                        # 1e-4
    WARMUP_EPOCHS = CFG.WARMUP_EPOCHS  # 5
    EMA_DECAY = CFG.EMA_DECAY          # 0.9996
    PATIENCE = CFG.PATIENCE            # 25
    DROP_PATH = CFG.DROP_PATH          # 0.2
    GAMMA = CFG.GAMMA                  # 2.5
    AUX_LAMBDA = CFG.AUX_LAMBDA        # 0.35
    NUM_WORKERS = CFG.NUM_WORKERS      # 8
    TTA = CFG.TTA                      # True
    CAM_PER_CLASS = CFG.CAM_PER_CLASS  # 20

# ---------- Output directories (separate from Stage 1) ----------
FT_DIR   = os.path.join(CFG.ROOT_DIR, "ft672")
FT_CKPT  = os.path.join(FT_DIR, "ckpt")
FT_FIG   = os.path.join(FT_DIR, "figs")
FT_REP   = os.path.join(FT_DIR, "reports")
FT_FEAT  = os.path.join(FT_DIR, "features")
FT_CAM   = os.path.join(FT_DIR, "cams")
for d in [FT_DIR, FT_CKPT, FT_FIG, FT_REP, FT_FEAT, FT_CAM]:
    os.makedirs(d, exist_ok=True)
print(f"[Fine-tuning] Output directory: {FT_DIR}")

# ---------- Data augmentation at 672 x 672 (matching Stage 1 transformations) ----------
train_tfms_ft = A.Compose([
    A.RandomResizedCrop(size=(FT.IMG_SIZE, FT.IMG_SIZE), scale=(0.90, 1.0), ratio=(0.9, 1.1)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.HueSaturationValue(10, 15, 10, p=0.3),
    A.CoarseDropout(max_holes=2, max_height=int(FT.IMG_SIZE*0.05), max_width=int(FT.IMG_SIZE*0.05), p=0.25),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])
val_tfms_ft = A.Compose([
    A.Resize(height=FT.IMG_SIZE, width=FT.IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

# ---------- Datasets and data loaders ----------
train_ds_ft = ISICDataset(train_df, CFG.IMAGE_DIR, transforms=train_tfms_ft)
val_ds_ft   = ISICDataset(val_df,   CFG.IMAGE_DIR, transforms=val_tfms_ft)
test_ds_ft  = ISICDataset(test_df,  CFG.IMAGE_DIR, transforms=val_tfms_ft)

class_counts = train_df['label'].value_counts().sort_index().values
class_weights_np = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights_np[train_df['label'].values]
sampler_ft = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader_ft = DataLoader(train_ds_ft, batch_size=FT.BATCH_SIZE, sampler=sampler_ft,
                             num_workers=FT.NUM_WORKERS, pin_memory=True)
val_loader_ft   = DataLoader(val_ds_ft,   batch_size=FT.BATCH_SIZE*2, shuffle=False,
                             num_workers=FT.NUM_WORKERS, pin_memory=True)
test_loader_ft  = DataLoader(test_ds_ft,  batch_size=FT.BATCH_SIZE*2, shuffle=False,
                             num_workers=FT.NUM_WORKERS, pin_memory=True)

# ---------- Model at 672 x 672 ----------
# Reuse CoAtNetWithAux, rebuilding its backbone for 672 x 672 inputs.
model_ft = CoAtNetWithAux(CFG.MODEL_NAME, CFG.NUM_CLASSES, drop_path_rate=FT.DROP_PATH).to(DEVICE)
# Some timm backbones have image-size-dependent positional or relative-position tables, so initialize the backbone at 672 x 672:
model_ft.backbone = timm.create_model(
    CFG.MODEL_NAME, pretrained=False, num_classes=0, global_pool='avg',
    drop_path_rate=FT.DROP_PATH, img_size=FT.IMG_SIZE
).to(DEVICE)
model_ft = model_ft.to(memory_format=torch.channels_last)

# ---------- Load the best Stage 1 EMA checkpoint and resize relative-position bias tables when possible ----------
import math as _m
from torch.nn import functional as _F

def _sqrt_int(n):
    s = int(round(_m.sqrt(n)));
    if s*s != n: return None
    return s

def _resize_rel_pos_table(weight, target_num_tokens):
    # weight: [L_old, nH] → [L_new, nH], L_new = S_new^2
    L_old, nH = weight.shape
    So = _sqrt_int(L_old); Sn = _sqrt_int(target_num_tokens)
    if So is None or Sn is None:  # Skip non-square token counts
        return None
    w = weight.transpose(0,1).contiguous().view(nH, So, So).unsqueeze(0)   # [1,nH,So,So]
    w = _F.interpolate(w, size=(Sn, Sn), mode="bicubic", align_corners=False)
    w = w.squeeze(0).contiguous().view(nH, Sn*Sn).transpose(0,1)           # [L_new, nH]
    return w

ckpt_path = os.path.join(CFG.CKPT_DIR, "best_model_stage1.pth")
state_src = torch.load(ckpt_path, map_location=DEVICE)
state_tgt = model_ft.state_dict()
resized = skipped = 0
for k in list(state_src.keys()):
    if k not in state_tgt:
        del state_src[k]; skipped += 1; continue
    v_src, v_tgt = state_src[k], state_tgt[k]
    if v_src.shape == v_tgt.shape:
        continue
    if "relative_position_bias_table" in k and v_src.ndim==2 and v_tgt.ndim==2:
        w_new = _resize_rel_pos_table(v_src, v_tgt.shape[0])
        if w_new is not None:
            state_src[k] = w_new; resized += 1
        else:
            del state_src[k]; skipped += 1
    else:
        del state_src[k]; skipped += 1

ld = model_ft.load_state_dict(state_src, strict=False)
print(f"[Fine-tuning] Checkpoint loading: resized_relpos={resized} | skipped={skipped} | missing={len(ld.missing_keys)} | unexpected={len(ld.unexpected_keys)}")

# ---------- EMA, optimizer, and learning-rate scheduler ----------
ema_ft = ModelEmaV2(model_ft, decay=FT.EMA_DECAY, device=DEVICE); ema_ft.module.load_state_dict(model_ft.state_dict())
optimizer_ft = torch.optim.AdamW(model_ft.parameters(), lr=FT.LR, weight_decay=FT.WD)
scaler_ft = GradScaler()

total_steps = len(train_loader_ft) * FT.EPOCHS
warmup_steps = len(train_loader_ft) * FT.WARMUP_EPOCHS
def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    prog = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * prog))
scheduler_ft = torch.optim.lr_scheduler.LambdaLR(optimizer_ft, lr_lambda)

# ---------- Loss functions: class-balanced focal loss and auxiliary losses ----------
alpha_eff_ft = effective_num_alpha(train_df['label'].values, num_classes=CFG.NUM_CLASSES)
criterion_warm_ft = FocalLoss(alpha=alpha_eff_ft * 0.3, gamma=FT.GAMMA)  # Stage 1 uses a warm-up loss weighting period
criterion_cb_ft   = FocalLoss(alpha=alpha_eff_ft,         gamma=FT.GAMMA)

T1 = CFG.DRW_SWITCH  # As in Stage 1, full class-balanced weighting is applied from epoch T1 onward
print(f"[Fine-tuning] Full class-balanced focal loss starts at epoch >= {T1}")

# ---------- Fine-tuning loop ----------
history_ft = {"train_loss": [], "val_loss": [], "val_f1": []}
best_f1, patience = -1.0, 0

for epoch in range(FT.EPOCHS):
    model_ft.train()
    running_loss = 0.0
    pbar = tqdm(train_loader_ft, desc=f"[FT 672] Epoch {epoch+1}/{FT.EPOCHS}")
    for imgs, labels in pbar:
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last)
        labels = labels.to(DEVICE)

        optimizer_ft.zero_grad(set_to_none=True)
        with autocast():
            outputs = model_ft(imgs)
            crit_now = criterion_cb_ft if epoch >= T1 else criterion_warm_ft
            loss, loss_main, loss_aux = total_loss(outputs, labels, crit_now, FT.AUX_LAMBDA)

        scaler_ft.scale(loss).backward()
        nn.utils.clip_grad_norm_(model_ft.parameters(), CFG.GRAD_CLIP)
        scaler_ft.step(optimizer_ft); scaler_ft.update()
        scheduler_ft.step()
        ema_ft.update(model_ft)

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader_ft)

    # ---- Validation using EMA weights ----
    model_ft.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader_ft:
            imgs = imgs.to(DEVICE, memory_format=torch.channels_last)
            labels = labels.to(DEVICE)
            with autocast():
                logits, _, _ = ema_ft.module(imgs)
                vloss = F.cross_entropy(logits, labels)
            val_loss += vloss.item()
            pred = logits.argmax(1)
            all_preds.extend(pred.cpu().numpy()); all_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader_ft)
    val_f1 = f1_score(np.array(all_labels), np.array(all_preds), average="macro")

    history_ft["train_loss"].append(train_loss)
    history_ft["val_loss"].append(val_loss)
    history_ft["val_f1"].append(val_f1)

    print(f"[FT {epoch+1:03d}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_macroF1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1, patience = val_f1, 0
        torch.save(ema_ft.module.state_dict(), os.path.join(FT_CKPT, "best_model_ft_672.pth"))
        print(f"   New best fine-tuning validation macro-F1: {best_f1:.4f}")
    else:
        patience += 1
        if patience >= FT.PATIENCE:
            print("   Fine-tuning early stopping triggered."); break

# ---------- Load the best fine-tuned checkpoint ----------
ema_ft.module.load_state_dict(torch.load(os.path.join(FT_CKPT, "best_model_ft_672.pth"), map_location=DEVICE))
best_ft = ema_ft.module.eval()

# ---------- Plotting, metrics, and prediction helpers ----------
def plot_and_save_cm(cm, labels, title, out_path):
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=labels, yticklabels=labels)
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(out_path, dpi=300); plt.close()

def plot_and_save_cm_norm(cm, labels, title, out_path):
    cmn = cm.astype(float)/np.maximum(cm.sum(axis=1, keepdims=True), 1)
    plt.figure(figsize=(8,6))
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='magma', xticklabels=labels, yticklabels=labels, vmin=0, vmax=1)
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(out_path, dpi=300); plt.close()

def plot_and_save_roc(y_true, y_prob, labels, out_path, tag=""):
    y_bin = label_binarize(y_true, classes=np.arange(len(labels)))
    fpr, tpr, roc_auc = {}, {}, {}; valid = []
    for i in range(len(labels)):
        pos = y_bin[:, i].sum(); neg = len(y_bin[:, i]) - pos
        if pos>0 and neg>0:
            fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i]); valid.append(i)
    plt.figure(figsize=(8,6))
    for i in valid: plt.plot(fpr[i], tpr[i], lw=1.4, label=f"{labels[i]} (AUC={roc_auc[i]:.3f})")
    plt.plot([0,1],[0,1],'k--',lw=1)
    try:
        micro_fpr, micro_tpr, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
        micro_auc = auc(micro_fpr, micro_tpr)
        plt.plot(micro_fpr, micro_tpr, lw=2.0, label=f"micro (AUC={micro_auc:.3f})")
    except ValueError: pass
    plt.title(f"ROC — {tag}"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(fontsize=8, ncol=2)
    plt.tight_layout(); plt.savefig(out_path, dpi=300); plt.close()

def metrics_table(y_true, y_prob):
    y_pred = y_prob.argmax(1)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        y_bin = label_binarize(y_true, classes=np.arange(CFG.NUM_CLASSES))
        auc_macro = roc_auc_score(y_bin, y_prob, average="macro")
        auc_weighted = roc_auc_score(y_bin, y_prob, average="weighted")
        auc_micro = roc_auc_score(y_bin.ravel(), y_prob.ravel())
    except Exception:
        auc_macro = auc_weighted = auc_micro = float("nan")
    return macro_f1, mcc, auc_micro, auc_macro, auc_weighted

@torch.no_grad()
def predict_loader(model, loader, tta=True):
    model.eval()
    probs_all, labels_all = [], []
    for imgs, labels in tqdm(loader, desc="Predict FT"):
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last)
        labels = labels.to(DEVICE)
        with autocast():
            if tta:
                batch_cat = torch.cat([imgs, torch.flip(imgs,[3]), torch.flip(imgs,[2])], dim=0)
                logits = model(batch_cat)[0]
                p0,p1,p2 = logits.chunk(3, dim=0)
                probs = (p0.softmax(1)+p1.softmax(1)+p2.softmax(1))/3.0
            else:
                probs = model(imgs)[0].softmax(1)
        probs_all.append(probs.float().cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    return np.concatenate(probs_all), np.array(labels_all)

# ---------- Validation and test evaluation ----------
probs_val_ft, labels_val_ft = predict_loader(best_ft, val_loader_ft, tta=FT.TTA)
probs_tst_ft, labels_tst_ft = predict_loader(best_ft, test_loader_ft, tta=FT.TTA)
preds_val_ft = probs_val_ft.argmax(1); preds_tst_ft = probs_tst_ft.argmax(1)

cr_val = classification_report(labels_val_ft, preds_val_ft, target_names=CFG.CLASS_NAMES, digits=4)
cr_tst = classification_report(labels_tst_ft, preds_tst_ft, target_names=CFG.CLASS_NAMES, digits=4)
cm_val = confusion_matrix(labels_val_ft, preds_val_ft)
cm_tst = confusion_matrix(labels_tst_ft, preds_tst_ft)

open(os.path.join(FT_REP, "classification_report_val.txt"), "w").write(cr_val)
open(os.path.join(FT_REP, "classification_report_test.txt"), "w").write(cr_tst)

plot_and_save_cm(cm_val, CFG.CLASS_NAMES, "Confusion Matrix (VAL, FT672)", os.path.join(FT_FIG, "cm_val_raw.png"))
plot_and_save_cm_norm(cm_val, CFG.CLASS_NAMES, "Confusion Matrix Normalized (VAL, FT672)", os.path.join(FT_FIG, "cm_val_norm.png"))
plot_and_save_roc(labels_val_ft, probs_val_ft, CFG.CLASS_NAMES, os.path.join(FT_FIG, "roc_val.png"), tag="VAL FT672")

plot_and_save_cm(cm_tst, CFG.CLASS_NAMES, "Confusion Matrix (TEST, FT672)", os.path.join(FT_FIG, "cm_test_raw.png"))
plot_and_save_cm_norm(cm_tst, CFG.CLASS_NAMES, "Confusion Matrix Normalized (TEST, FT672)", os.path.join(FT_FIG, "cm_test_norm.png"))
plot_and_save_roc(labels_tst_ft, probs_tst_ft, CFG.CLASS_NAMES, os.path.join(FT_FIG, "roc_test.png"), tag="TEST FT672")

f1_v, mcc_v, auc_micro_v, auc_macro_v, auc_weighted_v = metrics_table(labels_val_ft, probs_val_ft)
f1_t, mcc_t, auc_micro_t, auc_macro_t, auc_weighted_t = metrics_table(labels_tst_ft, probs_tst_ft)
pd.DataFrame([{"macro_f1": f1_v, "mcc": mcc_v, "auc_micro": auc_micro_v, "auc_macro": auc_macro_v, "auc_weighted": auc_weighted_v}])\
  .to_csv(os.path.join(FT_REP, "val_summary.csv"), index=False)
pd.DataFrame([{"macro_f1": f1_t, "mcc": mcc_t, "auc_micro": auc_micro_t, "auc_macro": auc_macro_t, "auc_weighted": auc_weighted_t, "TTA": FT.TTA}])\
  .to_csv(os.path.join(FT_REP, "test_summary.csv"), index=False)

# ---------- Temperature scaling: select T on validation data and apply it to test probabilities ----------
Ts = np.arange(0.5, 5.01, 0.1)
best_T, best_nll = 1.0, 1e9
eps = 1e-8
logp_val = np.log(np.maximum(probs_val_ft, eps))
for T in Ts:
    scaled_logits = (logp_val / T).clip(-50, 50)
    scaled_logits = scaled_logits - scaled_logits.max(axis=1, keepdims=True)
    scaled_probs = np.exp(scaled_logits)
    scaled_probs = scaled_probs / scaled_probs.sum(axis=1, keepdims=True)
    nll = log_loss(labels_val_ft, scaled_probs, labels=np.arange(CFG.NUM_CLASSES))
    if nll < best_nll: best_nll, best_T = nll, T

def expected_calibration_error(probs, labels, n_bins=15):
    bins = np.linspace(0,1,n_bins+1)
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    acc = (pred==labels).astype(float)
    ece = 0.0
    for i in range(n_bins):
        m = (conf>bins[i]) & (conf<=bins[i+1]); p = m.mean()
        if p>0: ece += abs(acc[m].mean() - conf[m].mean())*p
    return float(ece)

labels_tst_oh = np.eye(CFG.NUM_CLASSES)[labels_tst_ft]
probs_test_raw = probs_tst_ft.copy()
probs_test_cal = torch.tensor(np.log(np.maximum(probs_test_raw,eps))/best_T).softmax(dim=1).numpy()
ece_raw = expected_calibration_error(probs_test_raw, labels_tst_ft, 15)
ece_cal = expected_calibration_error(probs_test_cal, labels_tst_ft, 15)
brier_raw = float(np.mean((probs_test_raw - labels_tst_oh)**2))
brier_cal = float(np.mean((probs_test_cal - labels_tst_oh)**2))

json.dump({"best_T": float(best_T), "val_NLL": float(best_nll),
           "ECE_raw": ece_raw, "ECE_cal": ece_cal,
           "Brier_raw": brier_raw, "Brier_cal": brier_cal},
          open(os.path.join(FT_REP, "calibration.json"), "w"), indent=2)

# ---------- Feature export ----------
@torch.no_grad()
def export_features(m, loader, split_name, with_ids=False):
    m.eval(); feats, labels, ids = [], [], []
    for batch in tqdm(loader, desc=f"[FT] Export {split_name}"):
        if with_ids:
            imgs, lbls, idlist = batch
        else:
            imgs, lbls = batch; idlist=None
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last)
        with autocast(): _, _, feat = m(imgs)
        feats.append(feat.float().cpu().numpy())
        labels.extend(lbls.numpy())
        if with_ids and idlist is not None: ids += list(idlist)
    feats = np.concatenate(feats, axis=0)
    np.save(os.path.join(FT_FEAT, f"features_{split_name}.npy"), feats)
    meta = {"label": labels}
    if with_ids: meta["image"] = ids
    pd.DataFrame(meta).to_csv(os.path.join(FT_FEAT, f"meta_{split_name}.csv"), index=False)
    print(f"{split_name} features saved; shape={feats.shape}")

# Create loaders that also return image IDs
val_ds_eval  = ISICDataset(val_df,  CFG.IMAGE_DIR, transforms=val_tfms_ft,  return_id=True)
test_ds_eval = ISICDataset(test_df, CFG.IMAGE_DIR, transforms=val_tfms_ft,  return_id=True)
train_loader_eval = DataLoader(ISICDataset(train_df, CFG.IMAGE_DIR, transforms=val_tfms_ft, return_id=True),
                               batch_size=FT.BATCH_SIZE*2, shuffle=False, num_workers=FT.NUM_WORKERS, pin_memory=True)
val_loader_eval2  = DataLoader(val_ds_eval,  batch_size=FT.BATCH_SIZE*2, shuffle=False, num_workers=FT.NUM_WORKERS, pin_memory=True)
test_loader_eval2 = DataLoader(test_ds_eval, batch_size=FT.BATCH_SIZE*2, shuffle=False, num_workers=FT.NUM_WORKERS, pin_memory=True)

export_features(best_ft, train_loader_eval, "train", with_ids=True)
export_features(best_ft, val_loader_eval2,  "val",   with_ids=True)
export_features(best_ft, test_loader_eval2, "test",  with_ids=True)

# ---------- Grad-CAM++ examples ----------
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

def find_last_conv_layer(model):
    last_conv = None
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d): last_conv = m
    if last_conv is None:
        raise RuntimeError("No convolutional layer was found. Select a target layer manually for CAM.")
    return last_conv

target_layer = find_last_conv_layer(best_ft)

def tensor_to_img_np(t):
    tn = t.clone().float().cpu()
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    tn = (tn*std + mean).clamp(0,1)
    return np.transpose(tn.numpy(), (1,2,0))

class LogitsOnlyWrapperFT(nn.Module):
    """Expose only class logits to Grad-CAM implementations."""
    def __init__(self, wrapped_model):
        super().__init__()
        self.wrapped_model = wrapped_model

    def forward(self, x):
        return self.wrapped_model(x)[0]


def save_cams(split_name, loader, per_class=20):
    os.makedirs(os.path.join(FT_CAM, split_name), exist_ok=True)
    # Select the first N examples per ground-truth class
    probs_all, labels_all, ids_all = [], [], []
    for imgs, labels, ids in DataLoader(loader.dataset, batch_size=FT.BATCH_SIZE, shuffle=False):
        imgs = imgs.to(DEVICE)
        with autocast():
            logits, _, _ = best_ft(imgs)
            probs = logits.softmax(1).float().cpu().numpy()
        probs_all.append(probs)
        labels_all += list(labels.numpy())
        ids_all += list(ids)
    probs_all = np.concatenate(probs_all, axis=0)
    labels_all = np.array(labels_all)
    preds_all = probs_all.argmax(1)

    chosen = {k: [] for k in range(CFG.NUM_CLASSES)}
    for i, (y, p) in enumerate(zip(labels_all, preds_all)):
        c = int(y)
        if len(chosen[c]) < per_class:
            chosen[c].append(i)
        if all(len(v) >= per_class for v in chosen.values()):
            break

    cam_model_ft = LogitsOnlyWrapperFT(best_ft)
    cam = GradCAMPlusPlus(model=cam_model_ft, target_layers=[target_layer], use_cuda=(DEVICE == "cuda"))
    loader_plain = DataLoader(loader.dataset, batch_size=1, shuffle=False)
    idx_global = 0
    save_map = set(sum(chosen.values(), []))
    for (img, y, idstr) in tqdm(loader_plain, total=len(loader_plain), desc=f"CAM {split_name}"):
        if idx_global in save_map:
            img = img.to(DEVICE)
            with torch.no_grad(), autocast():
                logits, _, _ = best_ft(img)
                pred = logits.argmax(1).item()
            grayscale_cam = cam(input_tensor=img, targets=None)[0]
            rgb_np = tensor_to_img_np(img[0])
            vis = show_cam_on_image(rgb_np, grayscale_cam, use_rgb=True)
            cls_name = CFG.CLASS_NAMES[int(y)]
            out_dir = os.path.join(FT_CAM, split_name, cls_name)
            os.makedirs(out_dir, exist_ok=True)
            Image.fromarray(vis).save(os.path.join(out_dir, f"{idstr[0]}_pred-{CFG.CLASS_NAMES[pred]}.png"))
        idx_global += 1

save_cams("val",  val_loader_eval2,  per_class=FT.CAM_PER_CLASS)
save_cams("test", test_loader_eval2, per_class=FT.CAM_PER_CLASS)

print("\nStage 2 fine-tuning at 672 x 672 is complete.")
print(f"Checkpoint: {FT_CKPT}/best_model_ft_672.pth")
print(f"Figures: {FT_FIG}")
print(f"Reports: {FT_REP}")
print(f"Features: {FT_FEAT}")
print(f"CAM images: {FT_CAM}")
